# Reinforcement Learning (DQN)

## Setup

### Runtime setup

Set up the runtime by running exactly one code cell in this section.

#### Colab

In [ ]:
!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
!pip install -r dqn/requirements.txt

import sys

sys.path.insert(0, "dqn/src")

##### Optional: Mount Google Drive

Mount Google Drive by running this code cell once per Colab session.

In [ ]:
from pathlib import Path

from google.colab import drive

drive_dir = Path("/content/drive/MyDrive/rl_lab")

drive.mount("/content/drive")
drive_dir.mkdir(parents=True, exist_ok=True)

#### Local

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent.parent

sys.path.insert(0, str(repo_root / "dqn" / "src"))

### Training setup

Set up training by running exactly one code cell in this section.

#### CartPole

In [ ]:
from dqn.training import Trainer, TrainingConfig

ENV_ID = "CartPole-v1"
REPLAY_MEMORY_CAPACITY = 10_000
PLOT_Y_LABEL = "Duration"
MAX_STEPS=500
trainer_factory = Trainer

config = TrainingConfig(num_episodes=50)

#### LunarLander

In [ ]:
from dqn.training import Trainer, TrainingConfig

ENV_ID = "LunarLander-v3"
REPLAY_MEMORY_CAPACITY = 50_000
PLOT_Y_LABEL = "Return"
MAX_STEPS=1000
trainer_factory = Trainer

config = TrainingConfig(
    num_episodes=600,
    eps_start=1.0,
    eps_decay=50_000,
)

#### LunarLander with tuning

In [ ]:
from pathlib import Path

from dqn.tuned_training import TunedTrainer, TunedTrainingConfig

ENV_ID = "LunarLander-v3"
REPLAY_MEMORY_CAPACITY = 50_000
PLOT_Y_LABEL = "Return"
MAX_STEPS=1000
trainer_factory = TunedTrainer

LOCAL_CHECKPOINT_PATH = Path("best_checkpoint.pt")
DRIVE_CHECKPOINT_PATH = Path("/content/drive/MyDrive/rl_lab/best_checkpoint.pt")
# CHECKPOINT_PATH = LOCAL_CHECKPOINT_PATH
CHECKPOINT_PATH = DRIVE_CHECKPOINT_PATH

config = TunedTrainingConfig(
    num_episodes=600,
    eps_start=1.0,
    eps_decay=50_000,
    learning_starts=5_000,
    optimize_every=2,
    save_best_checkpoint=True,
    checkpoint_window=50,
    checkpoint_path=CHECKPOINT_PATH,
)

## Training

In [ ]:
import gymnasium as gym

from dqn.visualize import EpisodePlotter

env = gym.make(ENV_ID)

plotter = EpisodePlotter(y_label=PLOT_Y_LABEL)

trainer = trainer_factory(
    env,
    seed=42,
    replay_memory_capacity=REPLAY_MEMORY_CAPACITY,
)

### Optional: Load checkpoint

In [ ]:
from dqn.checkpointing import load_checkpoint

load_checkpoint(trainer, CHECKPOINT_PATH)

In [ ]:
try:
    result = trainer.train(config, plotter=plotter)
finally:
    env.close()

plotter.plot_returns(result.episode_returns, show_result=True)

## Visualization

In [ ]:
from dqn.visualize import record_episode, show_animation

frames, scores = record_episode(
    make_env=lambda: gym.make(ENV_ID, render_mode="rgb_array"),
    q_net=trainer.q_net,
    device=trainer.device,
    return_scores=True,
)

show_animation(frames, scores=scores)

## Optional: Sync checkpoint with Google Drive

### Copy checkpoint from Drive

In [ ]:
import shutil

shutil.copy2(DRIVE_CHECKPOINT_PATH, LOCAL_CHECKPOINT_PATH)

### Copy checkpoint to Drive

In [ ]:
import shutil

shutil.copy2(LOCAL_CHECKPOINT_PATH, DRIVE_CHECKPOINT_PATH)